In [ ]:
import pandas as pd, numpy as np, sys
from pathlib import Path
from unidecode import unidecode

root = Path.cwd().parent/"src"
if str(root) not in sys.path:
    sys.path.append(str(root))

pd.set_option('display.max_columns', None)
from routes.Paths import Ruta_opl, Ruta_global, Ruta_local, Ruta_nombreshomologos_agentes

In [ ]:
df_opl = pd.read_excel(Ruta_opl, sheet_name="Combinado")
df_homologacion = pd.read_excel(Ruta_nombreshomologos_agentes )
colmnas_opl = [
            "Identificacion", "ID Workday", "ID NICE", "Nombre", "Jefe Inmediato", "Lider", "Posicion",
            "Servicio", "Estado", "Fecha Inicio Servicio", "Grupo/Wave", "FECHA RETIRO", "Correo Corporativo",
             "Vista 360" ]
df_opl = df_opl[colmnas_opl]
df_opl = df_opl.drop_duplicates(["Identificacion", "ID Workday", "Correo Corporativo", "Vista 360"])

def clean_col(columnas):
    if isinstance(columnas, str):
        columnas = columnas.strip().title()
        columnas = unidecode(columnas)

        if columnas == "":
            return "-"
    return columnas

df_opl = df_opl.fillna("-")
df_opl["FECHA RETIRO"] = pd.to_datetime(df_opl["FECHA RETIRO"], format="%d/%m/%Y", errors="coerce")
df_opl["Fecha Inicio Servicio"] = pd.to_datetime(df_opl["Fecha Inicio Servicio"], format="%d/%m/%Y", errors="coerce")

col_limpias = ["Nombre", "Jefe Inmediato", "Lider", "Posicion", "Servicio", "Estado", "Grupo/Wave"]
for col in col_limpias:
    df_opl[col] = df_opl[col].apply(clean_col)

df_opl = df_opl.rename(columns={
    "Identificacion":"Documento",
    "ID Workday":"Id_Workday",
    "ID NICE":"Id_Nice",
    "Nombre":"Nombre_Agente",
    "Jefe Inmediato":"Supervisor",
    "Grupo/Wave":"Wave",
    "Correo Corporativo":"Correo_corporativo",
    "Vista 360":"Correo_contratista",
    "FECHA RETIRO": "Fecha_retiro",
    "Fecha Inicio Servicio":"Fecha_inicio_servicio"
})
df_opl = df_opl[~df_opl["Correo_corporativo"].isin(["-"])]
df_opl = df_opl.drop_duplicates(["Correo_contratista"])
category_column = ["Posicion", "Servicio", "Estado", "Wave"]
for col in category_column:
    df_opl[col] = df_opl[col].astype("category")
